# 01 Data Cleaning

**Project:** AI-Powered Customer Retention Decision Support System for E-Commerce  
**Dataset:** Online Retail II transaction data  
**Purpose:** Convert raw transaction records into clean, analysis-ready tables for customer retention modeling.

This notebook is intentionally written as a repeatable audit trail: it documents the cleaning decisions,
shows the effect of each filter, and validates the outputs that already exist in `data/processed`.

> **Data safety note:** this notebook does **not** overwrite files in `data/processed` unless
> `SAVE_OUTPUTS` is manually changed to `True`.

## Cleaning Decisions

The raw transaction file contains invoice-level sales records. For retention modeling, the project
needs a clean purchase table and a separate view of returns/cancellations.

Key decisions:

1. Keep only rows with a valid `CustomerID` for customer-level modeling.
2. Standardize column names so later notebooks can use stable field names.
3. Treat invoices beginning with `C` and rows with non-positive quantity or price as return/cancellation
   or invalid sales activity.
4. Keep return/cancellation records separately for future customer behavior features.
5. Build `cleaned_transactions` from positive, customer-linked sales rows only.

## 1. Setup

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

sns.set_theme(
    style="whitegrid",
    context="notebook",
    palette="Set2",
    rc={
        "figure.figsize": (11, 5),
        "axes.titleweight": "bold",
        "axes.titlesize": 14,
        "axes.labelsize": 11,
    },
)

In [ ]:
# Resolve project paths from the notebook location.
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = (
    NOTEBOOK_DIR.parent
    if NOTEBOOK_DIR.name == "notebooks"
    else Path(r"C:/Learning/BANA8083/AI-retention-decision-support")
)

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "online_retail_II.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

CLEANED_TRANSACTIONS_PATH = PROCESSED_DIR / "cleaned_transactions.csv"
RETURNS_CANCELLATIONS_PATH = PROCESSED_DIR / "returns_cancellations.csv"
CLEANING_SUMMARY_PATH = PROCESSED_DIR / "data_cleaning_summary.csv"

# Keep this False unless you intentionally want to regenerate the saved processed files.
SAVE_OUTPUTS = False

paths = pd.DataFrame(
    {
        "asset": [
            "raw data",
            "processed directory",
            "cleaned transactions",
            "returns/cancellations",
            "cleaning summary",
        ],
        "path": [
            RAW_DATA_PATH,
            PROCESSED_DIR,
            CLEANED_TRANSACTIONS_PATH,
            RETURNS_CANCELLATIONS_PATH,
            CLEANING_SUMMARY_PATH,
        ],
        "exists": [
            RAW_DATA_PATH.exists(),
            PROCESSED_DIR.exists(),
            CLEANED_TRANSACTIONS_PATH.exists(),
            RETURNS_CANCELLATIONS_PATH.exists(),
            CLEANING_SUMMARY_PATH.exists(),
        ],
    }
)
paths

## 2. Load Raw Data

In [ ]:
raw = pd.read_csv(RAW_DATA_PATH)

print(f"Raw rows: {raw.shape[0]:,}")
print(f"Raw columns: {raw.shape[1]:,}")
raw.head()

In [ ]:
raw.info()

## 3. Standardize Columns and Types

The raw file uses names like `Invoice`, `Price`, and `Customer ID`. The cleaned project tables use
consistent names that are easier to reference in modeling and dashboard code.

In [ ]:
column_map = {
    "Invoice": "InvoiceNo",
    "Price": "UnitPrice",
    "Customer ID": "CustomerID",
}

df = raw.rename(columns=column_map).copy()

# Keep invoice and stock codes as strings because they can contain letters.
df["InvoiceNo"] = df["InvoiceNo"].astype(str).str.strip()
df["StockCode"] = df["StockCode"].astype(str).str.strip()
df["Description"] = df["Description"].astype("string").str.strip()
df["Country"] = df["Country"].astype("string").str.strip()

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
df["CustomerID"] = pd.to_numeric(df["CustomerID"], errors="coerce")

# Remove exact duplicate transaction rows before cleaning metrics are calculated.
df = df.drop_duplicates().reset_index(drop=True)

print(f"Rows after duplicate removal: {df.shape[0]:,}")
df.head()

## 4. Data Quality Audit

This audit shows the records that cannot be used directly as positive purchase transactions. Some
excluded rows are still useful later as behavioral signals, especially cancellations and returns.

In [ ]:
quality_checks = pd.DataFrame(
    {
        "check": [
            "Rows after duplicates removed",
            "Missing CustomerID",
            "Missing InvoiceDate",
            "Missing Quantity",
            "Missing UnitPrice",
            "Cancelled invoices",
            "Negative or zero Quantity",
            "Negative or zero UnitPrice",
        ],
        "count": [
            len(df),
            df["CustomerID"].isna().sum(),
            df["InvoiceDate"].isna().sum(),
            df["Quantity"].isna().sum(),
            df["UnitPrice"].isna().sum(),
            df["InvoiceNo"].str.startswith("C", na=False).sum(),
            (df["Quantity"] <= 0).sum(),
            (df["UnitPrice"] <= 0).sum(),
        ],
    }
)
quality_checks["share_of_rows"] = quality_checks["count"] / len(df)
quality_checks.style.format({"count": "{:,.0f}", "share_of_rows": "{:.2%}"})

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
plot_data = quality_checks.loc[quality_checks["check"] != "Rows after duplicates removed"].copy()
sns.barplot(data=plot_data, y="check", x="count", ax=ax, color="#4C78A8")
ax.set_title("Data Quality Issues Identified in Raw Transactions")
ax.set_xlabel("Rows")
ax.set_ylabel("")
ax.bar_label(ax.containers[0], fmt="{:,.0f}", padding=3)
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

## 5. Separate Returns and Cancellations

Returns and cancellations should not be counted as positive purchases, but they are valuable for later
customer retention features such as return rate and cancellation frequency.

In [ ]:
df["Revenue"] = df["Quantity"] * df["UnitPrice"]
df["IsCancelled"] = df["InvoiceNo"].str.startswith("C", na=False)
df["IsReturnOrCancel"] = df["IsCancelled"] | (df["Quantity"] <= 0) | (df["UnitPrice"] <= 0)

customer_linked = df[df["CustomerID"].notna()].copy()
customer_linked["CustomerID"] = customer_linked["CustomerID"].astype(int)

returns_cancellations = customer_linked[customer_linked["IsReturnOrCancel"]].copy()
returns_cancellations = returns_cancellations.sort_values("InvoiceDate").reset_index(drop=True)

print(f"Return/cancellation rows: {len(returns_cancellations):,}")
print(f"Customers with returns/cancellations: {returns_cancellations['CustomerID'].nunique():,}")
returns_cancellations.head()

## 6. Build Clean Positive Sales Table

The cleaned transaction table keeps only valid, customer-linked purchase activity. This table becomes
the base for customer-level feature engineering in the next notebook.

In [ ]:
cleaned_transactions = customer_linked[
    customer_linked["CustomerID"].notna()
    & customer_linked["InvoiceDate"].notna()
    & (customer_linked["Quantity"] > 0)
    & (customer_linked["UnitPrice"] > 0)
    & (~customer_linked["IsCancelled"])
].copy()

cleaned_transactions["Description"] = cleaned_transactions["Description"].fillna("Unknown")
cleaned_transactions = cleaned_transactions.sort_values(
    ["InvoiceDate", "InvoiceNo", "StockCode"]
).reset_index(drop=True)

print(f"Cleaned sales rows: {len(cleaned_transactions):,}")
print(f"Unique customers: {cleaned_transactions['CustomerID'].nunique():,}")
print(f"Total revenue: ${cleaned_transactions['Revenue'].sum():,.2f}")
print(
    "Date range: "
    f"{cleaned_transactions['InvoiceDate'].min()} to {cleaned_transactions['InvoiceDate'].max()}"
)

cleaned_transactions.head()

In [ ]:
cleaning_summary = pd.DataFrame(
    [
        ("raw_rows_after_duplicates_removed", len(df)),
        ("missing_customer_id", df["CustomerID"].isna().sum()),
        ("missing_invoice_date", df["InvoiceDate"].isna().sum()),
        ("missing_quantity", df["Quantity"].isna().sum()),
        ("missing_unit_price", df["UnitPrice"].isna().sum()),
        ("cancelled_invoices", df["IsCancelled"].sum()),
        ("negative_or_zero_quantity", (df["Quantity"] <= 0).sum()),
        ("negative_or_zero_price", (df["UnitPrice"] <= 0).sum()),
        ("returns_cancellations_rows", len(returns_cancellations)),
        ("customers_with_returns_cancellations", returns_cancellations["CustomerID"].nunique()),
        ("cleaned_sales_rows", len(cleaned_transactions)),
        ("cleaned_unique_customers", cleaned_transactions["CustomerID"].nunique()),
        ("cleaned_total_revenue", cleaned_transactions["Revenue"].sum()),
        ("cleaned_start_date", cleaned_transactions["InvoiceDate"].min()),
        ("cleaned_end_date", cleaned_transactions["InvoiceDate"].max()),
    ],
    columns=["check", "count"],
)

cleaning_summary

## 7. Validate Against Saved Processed Files

This section reads the existing files in `data/processed` and compares them to the regenerated
in-memory tables. It is a safety check only; it does not write anything.

In [ ]:
existing_summary = pd.read_csv(CLEANING_SUMMARY_PATH)
existing_cleaned = pd.read_csv(CLEANED_TRANSACTIONS_PATH, parse_dates=["InvoiceDate"])
existing_returns = pd.read_csv(RETURNS_CANCELLATIONS_PATH, parse_dates=["InvoiceDate"])

# Rebuild the in-memory summary here so section 7 can run cleanly even after a kernel restart
# or if the previous summary-display cell was skipped.
cleaning_summary = pd.DataFrame(
    [
        ("raw_rows_after_duplicates_removed", len(df)),
        ("missing_customer_id", df["CustomerID"].isna().sum()),
        ("missing_invoice_date", df["InvoiceDate"].isna().sum()),
        ("missing_quantity", df["Quantity"].isna().sum()),
        ("missing_unit_price", df["UnitPrice"].isna().sum()),
        ("cancelled_invoices", df["IsCancelled"].sum()),
        ("negative_or_zero_quantity", (df["Quantity"] <= 0).sum()),
        ("negative_or_zero_price", (df["UnitPrice"] <= 0).sum()),
        ("returns_cancellations_rows", len(returns_cancellations)),
        ("customers_with_returns_cancellations", returns_cancellations["CustomerID"].nunique()),
        ("cleaned_sales_rows", len(cleaned_transactions)),
        ("cleaned_unique_customers", cleaned_transactions["CustomerID"].nunique()),
        ("cleaned_total_revenue", cleaned_transactions["Revenue"].sum()),
        ("cleaned_start_date", cleaned_transactions["InvoiceDate"].min()),
        ("cleaned_end_date", cleaned_transactions["InvoiceDate"].max()),
    ],
    columns=["check", "count"],
)

validation = pd.DataFrame(
    {
        "artifact": ["cleaned_transactions", "returns_cancellations", "cleaning_summary"],
        "existing_rows": [len(existing_cleaned), len(existing_returns), len(existing_summary)],
        "regenerated_rows": [len(cleaned_transactions), len(returns_cancellations), len(cleaning_summary)],
    }
)
validation["matches"] = validation["existing_rows"] == validation["regenerated_rows"]
validation

In [ ]:
summary_compare = existing_summary.merge(
    cleaning_summary,
    on="check",
    how="outer",
    suffixes=("_existing", "_regenerated"),
)
summary_compare["matches"] = (
    summary_compare["count_existing"].astype(str)
    == summary_compare["count_regenerated"].astype(str)
)
summary_compare

## 8. Quick Visual Review of Cleaned Transactions

These charts are not a full exploratory analysis. They are quick checks that the cleaned sales table
behaves as expected before feature engineering begins.

In [ ]:
monthly_sales = (
    cleaned_transactions
    .assign(month=cleaned_transactions["InvoiceDate"].dt.to_period("M").dt.to_timestamp())
    .groupby("month", as_index=False)
    .agg(
        revenue=("Revenue", "sum"),
        invoices=("InvoiceNo", "nunique"),
        customers=("CustomerID", "nunique"),
    )
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=monthly_sales, x="month", y="revenue", marker="o", linewidth=2.5, ax=ax)
ax.set_title("Monthly Revenue After Cleaning")
ax.set_xlabel("")
ax.set_ylabel("Revenue")
ax.yaxis.set_major_formatter(lambda value, _: f"${value/1_000_000:.1f}M")
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
top_countries = (
    cleaned_transactions
    .groupby("Country", as_index=False)
    .agg(revenue=("Revenue", "sum"), customers=("CustomerID", "nunique"))
    .sort_values("revenue", ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=top_countries, x="revenue", y="Country", ax=ax, color="#59A14F")
ax.set_title("Top 10 Countries by Cleaned Revenue")
ax.set_xlabel("Revenue")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(lambda value, _: f"${value/1_000_000:.1f}M")
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

top_countries.style.format({"revenue": "${:,.0f}", "customers": "{:,.0f}"})

In [ ]:
customer_revenue = (
    cleaned_transactions
    .groupby("CustomerID", as_index=False)
    .agg(revenue=("Revenue", "sum"), invoices=("InvoiceNo", "nunique"))
    .sort_values("revenue", ascending=False)
)

fig, ax = plt.subplots(figsize=(11, 5))
sns.histplot(customer_revenue["revenue"], bins=60, ax=ax, color="#F28E2B")
ax.set_title("Distribution of Customer Revenue")
ax.set_xlabel("Customer Revenue")
ax.set_ylabel("Customers")
ax.set_xlim(0, customer_revenue["revenue"].quantile(0.99))
ax.xaxis.set_major_formatter(lambda value, _: f"${value:,.0f}")
sns.despine()
plt.tight_layout()
plt.show()

customer_revenue.head(10).style.format({"revenue": "${:,.0f}", "invoices": "{:,.0f}"})

## 9. Optional Export

The existing processed files are protected by default. If you intentionally want to overwrite them
after reviewing the regenerated tables, change `SAVE_OUTPUTS` to `True` in the setup cell and rerun
this cell.

In [ ]:
if SAVE_OUTPUTS:
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    cleaned_transactions.to_csv(CLEANED_TRANSACTIONS_PATH, index=False)
    returns_cancellations.to_csv(RETURNS_CANCELLATIONS_PATH, index=False)
    cleaning_summary.to_csv(CLEANING_SUMMARY_PATH, index=False)
    print("Processed files were overwritten because SAVE_OUTPUTS=True.")
else:
    print("SAVE_OUTPUTS=False, so no processed files were written or overwritten.")

## 10. Output Readiness

The cleaned data is ready for the next stage when the validation tables match and the quick visual
checks look reasonable.

Recommended next notebook: `02_feature_engineering.ipynb`

Next tasks:

1. Aggregate transactions to the customer level.
2. Create RFM features: recency, frequency, and monetary value.
3. Add behavioral features such as product diversity, customer lifetime, average order value, and
   return/cancellation rate.
4. Define the inactivity label for retention modeling.